# Run a test client

## Apply the feature store definitions

We'll mount the `feature_store.yaml` ConfigMap created by the Operator within a Kubernetes `Deployment` to run `feast apply`.

Then we create the client `Deployment` to apply the definitions, according to the [client.yaml](./client.yaml) manifest

We are going to emulate the `feast init -t postgres sample` command using Python code in an initContainer at client `Deployment` runtime. This is needed to mock the request of additional
parameters to configure the DB connection and also request the upload of example data to Postgres tables.

In [ ]:
!kubectl apply -f client.yaml

Monitoring the log of the `Deployment` and checking DB for populated data.

In [1]:
!kubectl wait --for=condition=available deployment/client --timeout=2m
!kubectl logs deploy/client -c feast-apply
!kubectl exec deploy/postgres -- psql -h localhost -U feast feast -c 'select count(*) from sample_driver_hourly_stats'

deployment.apps/client condition met
Starting feast initialization job...

Creating a new Feast repository in /feast-init/sample.


Executing feast apply...
/opt/conda/lib/python3.11/site-packages/feast/feature_store.py:575: RuntimeWarning: On demand feature view is an experimental feature. This API is stable, but the functionality does not scale well for offline retrieval
  warnings.warn(
No project found in the repository. Using project name sample defined in feature_store.yaml
Applying changes for project sample
Deploying infrastructure for driver_hourly_stats
Deploying infrastructure for driver_hourly_stats_fresh
Materializing 2 feature views to 2024-12-12 15:54:27+00:00 into the postgres online store.

driver_hourly_stats from 2024-12-12 15:39:32+00:00 to 2024-12-12 15:54:27+00:00:
0it [00:00, ?it/s]
driver_hourly_stats_fresh from 2024-12-12 15:39:32+00:00 to 2024-12-12 15:54:27+00:00:
0it [00:00, ?it/s]
 count 
-------
    15
(1 row)



Verify the client `feature_store.yaml`.

In [2]:
!kubectl exec deploy/client -c client -- cat work/sample/feature_repo/feature_store.yaml

project: sample
provider: local
offline_store:
    host: feast-example-offline.feast.svc.cluster.local
    type: remote
    port: 80
online_store:
    path: http://feast-example-online.feast.svc.cluster.local:80
    type: remote
registry:
    path: feast-example-registry.feast.svc.cluster.local:80
    registry_type: remote
auth:
    type: no_auth
entity_key_serialization_version: 3


## Execute feast commands inside the client Pod

In [5]:
!kubectl exec -i -t deploy/client -c client -- /bin/bash 
!ls -la
#!cd work/sample/
#!feast feature-views list
#!feast describe driver_hourly_stats
#!feast entities list
#!feast data-sources list

OSError: Background processes not supported.

## Run test code inside the client Pod

Finally, we run the full test suite from the client folder.

In [ ]:
!kubectl exec deploy/client -c client -- python3 work/sample/feature_repo/test_workflow.py

## Connect from your workstation

First we setup port forwarding to the feast services.

In [ ]:
!kubectl logs deploy/client -c client
!kubectl port-forward deploy/client 8888:8888